# Car Crash Simulation

In [5]:
from vpython import *
from math import sqrt

scene = canvas()
scene.autoscale = True

# Road
road = box(pos=vec(-35, -0.8, 0), size=vec(200, 0.2, 14), color=color.gray(0.4))

# Wall
Wall = box(pos=vec(15, 0, 0), size=vec(0.5, 3, 14), color=color.red)

# Cars
Car_Steel    = box(pos=vec(-80, 0, -4), size=vec(4.5, 1.5, 1.8), color=color.green)
Car_Aluminum = box(pos=vec(-80, 0,  0), size=vec(4.5, 1.5, 1.8), color=color.cyan)
Car_Carbon   = box(pos=vec(-80, 0,  4), size=vec(4.5, 1.5, 1.8), color=color.yellow)

car_Speed_mph = 50
car_Speed_mps = car_Speed_mph * 0.44704  # m/s

acceleration = True
accel = 3.0  # m/s^2

mass_car = 1500    # kg
mass_person = 75   # kg

youngs_Steel        = 200e9  # Pa
youngs_Aluminum     = 69e9   # Pa
youngs_Carbon_Fiber = 150e9  # Pa

L0_Steel    = Car_Steel.size.x
L0_Aluminum = Car_Aluminum.size.x
L0_Carbon   = Car_Carbon.size.x

# crumple zone derived from stiffness relative to aluminum as baseline
crumple_Steel    = L0_Steel    * (youngs_Aluminum / youngs_Steel)
crumple_Aluminum = L0_Aluminum * (youngs_Aluminum / youngs_Aluminum)
crumple_Carbon   = L0_Carbon   * (youngs_Aluminum / youngs_Carbon_Fiber)

print("Crumple zones (physics):")
print("Steel:", round(crumple_Steel, 3), "m")
print("Aluminum:", round(crumple_Aluminum, 3), "m")
print("Carbon Fiber:", round(crumple_Carbon, 3), "m")

for Car in [Car_Steel, Car_Aluminum, Car_Carbon]:
    Car.velocity = vector(car_Speed_mps, 0, 0)

dt = 0.005
t = 0
crashed_Steel = crashed_Aluminum = crashed_Carbon = False

# VISUAL EXAGGERATION — make it obvious even at high speeds
# Real deformation is ~0.5-1 cm, so we multiply by 80-100x for visual effect
visual_exaggeration = 100.0  

# Store original lengths for gradual crumple animation
original_lengths = {
    Car_Steel: L0_Steel,
    Car_Aluminum: L0_Aluminum,
    Car_Carbon: L0_Carbon
}

while t < 10:
    rate(100)

    if acceleration:
        for Car in [Car_Steel, Car_Aluminum, Car_Carbon]:
            if Car.velocity.x > 0:
                Car.velocity.x += accel * dt

    # Steel
    if Car_Steel.pos.x + Car_Steel.size.x/2 >= Wall.pos.x - Wall.size.x/2:
        if not crashed_Steel:
            v_impact = Car_Steel.velocity.x
            X_real = sqrt((mass_car * v_impact**2 * L0_Steel) / (2 * Car_Steel.size.y * Car_Steel.size.z * youngs_Steel))
            
            # Make visual crumple VERY obvious
            X_visual = min(X_real * visual_exaggeration, original_lengths[Car_Steel] * 0.95)  # Crumple up to 95% of car length
            Car_Steel.size.x = max(original_lengths[Car_Steel] - X_visual, 0.3)  # Don't go below 0.3m
            Car_Steel.pos.x = (Wall.pos.x - Wall.size.x/2) - Car_Steel.size.x/2
            
            delta_t = (2 * crumple_Steel) / v_impact
            a = v_impact / delta_t
            g_force = a / 9.81
            
            print("\n" + "="*50)
            print("STEEL CAR")
            print("deformation:", round(X_real*100, 3), "cm")
            print("Visual crumple shown:", round(X_visual, 2), "m")
            print("Impact time:", round(delta_t, 4), "s")
            print("G-force:", round(g_force, 2), "g")
            print("Result:", "SURVIVABLE" if g_force <= 40 else "FATAL")
            print("="*50)
            crashed_Steel = True
        Car_Steel.velocity = vector(0, 0, 0)
        # Continue crumpling gradually for effect
        if not crashed_Steel:
            Car_Steel.size.x *= 0.98

    # Aluminum
    if Car_Aluminum.pos.x + Car_Aluminum.size.x/2 >= Wall.pos.x - Wall.size.x/2:
        if not crashed_Aluminum:
            v_impact = Car_Aluminum.velocity.x
            X_real = sqrt((mass_car * v_impact**2 * L0_Aluminum) / (2 * Car_Aluminum.size.y * Car_Aluminum.size.z * youngs_Aluminum))
            
            X_visual = min(X_real * visual_exaggeration, original_lengths[Car_Aluminum] * 0.95)
            Car_Aluminum.size.x = max(original_lengths[Car_Aluminum] - X_visual, 0.3)
            Car_Aluminum.pos.x = (Wall.pos.x - Wall.size.x/2) - Car_Aluminum.size.x/2
            
            delta_t = (2 * crumple_Aluminum) / v_impact
            a = v_impact / delta_t
            g_force = a / 9.81
            
            print("\n" + "="*50)
            print("ALUMINUM CAR")
            print("deformation:", round(X_real*100, 3), "cm")
            print("Visual crumple shown:", round(X_visual, 2), "m")
            print("Impact time:", round(delta_t, 4), "s")
            print("G-force:", round(g_force, 2), "g")
            print("Result:", "SURVIVABLE" if g_force <= 40 else "FATAL")
            print("="*50)
            crashed_Aluminum = True
        Car_Aluminum.velocity = vector(0, 0, 0)

    # Carbon Fiber
    if Car_Carbon.pos.x + Car_Carbon.size.x/2 >= Wall.pos.x - Wall.size.x/2:
        if not crashed_Carbon:
            v_impact = Car_Carbon.velocity.x
            X_real = sqrt((mass_car * v_impact**2 * L0_Carbon) / (2 * Car_Carbon.size.y * Car_Carbon.size.z * youngs_Carbon_Fiber))
            
            X_visual = min(X_real * visual_exaggeration, original_lengths[Car_Carbon] * 0.95)
            Car_Carbon.size.x = max(original_lengths[Car_Carbon] - X_visual, 0.3)
            Car_Carbon.pos.x = (Wall.pos.x - Wall.size.x/2) - Car_Carbon.size.x/2
            
            delta_t = (2 * crumple_Carbon) / v_impact
            a = v_impact / delta_t
            g_force = a / 9.81
            
            print("\n" + "="*50)
            print("CARBON FIBER CAR ")
            print("deformation:", round(X_real*100, 3), "cm")
            print("Visual crumple shown:", round(X_visual, 2), "m")
            print("Impact time:", round(delta_t, 4), "s")
            print("G-force:", round(g_force, 2), "g")
            print("Result:", "SURVIVABLE" if g_force <= 40 else "FATAL")
            print("="*50)
            crashed_Carbon = True
        Car_Carbon.velocity = vector(0, 0, 0)

    for Car in [Car_Steel, Car_Aluminum, Car_Carbon]:
        Car.pos += Car.velocity * dt
    t += dt

<IPython.core.display.Javascript object>

Crumple zones (physics):
Steel: 1.552 m
Aluminum: 4.5 m
Carbon Fiber: 2.07 m

STEEL CAR
deformation: 0.257 cm
Visual crumple shown: 0.26 m
Impact time: 0.0956 s
G-force: 34.66 g
Result: SURVIVABLE

ALUMINUM CAR
deformation: 0.437 cm
Visual crumple shown: 0.44 m
Impact time: 0.277 s
G-force: 11.96 g
Result: SURVIVABLE

CARBON FIBER CAR 
deformation: 0.297 cm
Visual crumple shown: 0.3 m
Impact time: 0.1274 s
G-force: 25.99 g
Result: SURVIVABLE


# Car Crash Analysis

In [6]:
from vpython import *
from math import sqrt

# Create canvas for graphs
graph_canvas = canvas(title="Crash Test Analysis", width=1200, height=800, x=0, y=0)

# Material properties
mass_car = 1500  # kg

youngs_Steel = 200e9  # Pa
youngs_Aluminum = 69e9  # Pa
youngs_Carbon_Fiber = 150e9  # Pa

# Reference dimensions (based on typical car dimensions)
car_height = 1.5  # m
car_width = 1.8   # m
L0 = 4.5  # m (original car length)

# crumple zones
crumple_Steel = L0 * (youngs_Aluminum / youngs_Steel)
crumple_Aluminum = L0 * (youngs_Aluminum / youngs_Aluminum)
crumple_Carbon = L0 * (youngs_Aluminum / youngs_Carbon_Fiber)

print("="*60)
print("MATERIAL PROPERTIES")
print("="*60)
print(f"Young's Modulus - Steel: {youngs_Steel/1e9:.0f} GPa")
print(f"Young's Modulus - Aluminum: {youngs_Aluminum/1e9:.0f} GPa")
print(f"Young's Modulus - Carbon Fiber: {youngs_Carbon_Fiber/1e9:.0f} GPa")
print("\nCrumple Zones:")
print(f"Steel: {round(crumple_Steel, 3)} m")
print(f"Aluminum: {round(crumple_Aluminum, 3)} m")
print(f"Carbon Fiber: {round(crumple_Carbon, 3)} m")

# Test different speeds
speeds_mph = list(range(10, 161, 10))  # 10 to 160 mph in 10 mph steps
speeds_mps = [s * 0.44704 for s in speeds_mph]

# Store results for graphs
results = {
    'Steel': {'deformation': [], 'gforces': []},
    'Aluminum': {'deformation': [], 'gforces': []},
    'Carbon Fiber': {'deformation': [], 'gforces': []}
}

# Run calculations at different speeds
for speed_mph, speed_mps in zip(speeds_mph, speeds_mps):
    v_impact = speed_mps
    
    # Steel
    X_steel = sqrt((mass_car * v_impact**2 * L0) / (2 * car_height * car_width * youngs_Steel))
    delta_t_steel = (2 * crumple_Steel) / v_impact if v_impact > 0 else 0.001
    a_steel = v_impact / delta_t_steel if delta_t_steel > 0 else 0
    g_steel = a_steel / 9.81
    results['Steel']['deformation'].append(X_steel * 100)  # in cm
    results['Steel']['gforces'].append(g_steel)
    
    # Aluminum
    X_alu = sqrt((mass_car * v_impact**2 * L0) / (2 * car_height * car_width * youngs_Aluminum))
    delta_t_alu = (2 * crumple_Aluminum) / v_impact if v_impact > 0 else 0.001
    a_alu = v_impact / delta_t_alu if delta_t_alu > 0 else 0
    g_alu = a_alu / 9.81
    results['Aluminum']['deformation'].append(X_alu * 100)
    results['Aluminum']['gforces'].append(g_alu)
    
    # Carbon Fiber
    X_carbon = sqrt((mass_car * v_impact**2 * L0) / (2 * car_height * car_width * youngs_Carbon_Fiber))
    delta_t_carbon = (2 * crumple_Carbon) / v_impact if v_impact > 0 else 0.001
    a_carbon = v_impact / delta_t_carbon if delta_t_carbon > 0 else 0
    g_carbon = a_carbon / 9.81
    results['Carbon Fiber']['deformation'].append(X_carbon * 100)
    results['Carbon Fiber']['gforces'].append(g_carbon)

# Create deformation graph
deformation_graph = graph(width=600, height=350, 
                          title='Deformation vs Speed', 
                          xtitle='Speed (mph)', 
                          ytitle='Deformation (cm)',
                          foreground=color.black,
                          background=color.white)

steel_curve = gcurve(color=color.green, width=3, label='Steel')
aluminum_curve = gcurve(color=color.cyan, width=3, label='Aluminum')
carbon_curve = gcurve(color=color.yellow, width=3, label='Carbon Fiber')

for i, speed in enumerate(speeds_mph):
    steel_curve.plot(speed, results['Steel']['deformation'][i])
    aluminum_curve.plot(speed, results['Aluminum']['deformation'][i])
    carbon_curve.plot(speed, results['Carbon Fiber']['deformation'][i])

# Create g-force graph with threshold
gforce_graph = graph(width=600, height=350, 
                     title='G-Force vs Speed (40g Survivability Threshold)', 
                     xtitle='Speed (mph)', 
                     ytitle='G-Force (g)',
                     foreground=color.black,
                     background=color.white)

steel_gcurve = gcurve(color=color.green, width=3, label='Steel')
aluminum_gcurve = gcurve(color=color.cyan, width=3, label='Aluminum')
carbon_gcurve = gcurve(color=color.yellow, width=3, label='Carbon Fiber')

# Add 40g threshold line
threshold_line = gcurve(color=color.red, width=2, label='40g Survivability Limit')
for speed in speeds_mph:
    threshold_line.plot(speed, 40)

for i, speed in enumerate(speeds_mph):
    steel_gcurve.plot(speed, results['Steel']['gforces'][i])
    aluminum_gcurve.plot(speed, results['Aluminum']['gforces'][i])
    carbon_gcurve.plot(speed, results['Carbon Fiber']['gforces'][i])



<IPython.core.display.Javascript object>

MATERIAL PROPERTIES
Young's Modulus - Steel: 200 GPa
Young's Modulus - Aluminum: 69 GPa
Young's Modulus - Carbon Fiber: 150 GPa

Crumple Zones:
Steel: 1.552 m
Aluminum: 4.5 m
Carbon Fiber: 2.07 m
